# Python Foundations for Mathematical Code

**Mathematics for Machine Learning from First Principles · Lesson 1**

Machine-learning ideas are written in mathematics and carried out in code. This notebook shows how everyday Python (numbers, expressions, functions, loops, lists and classes) maps onto mathematical notation, and where a computer's arithmetic differs from arithmetic on paper. The later lessons build vectors, matrices and probability on exactly these foundations.

## Learning objectives

By the end of this notebook you will be able to:

1. explain the difference between Python's exact integers and its approximate floating-point numbers;
2. translate formulas such as $f(x) = 3x^2 - 2x + 1$, $\sum_{i=1}^{n} i$ and $n!$ into functions, loops and recursion;
3. represent a vector as a list and a matrix as a list of rows, and check that a matrix is rectangular;
4. write a small class that gives a 2-D vector the notation $\mathbf{u} + \mathbf{v}$ and $c\,\mathbf{v}$;
5. recognise representation, accumulation and cancellation errors, and handle them with `math.isclose` and `math.fsum`.

## Dependencies

**Tier 0 — Python standard library only.** The core cells import only standard-library modules: `math`, `sys`, `statistics`, `fractions`, `decimal`, `copy` and `numbers`. One optional comparison cell uses NumPy. It is labelled *Optional*, tagged `comparison`, checks that NumPy is installed before using it, and nothing after it depends on it, so you may skip it.

The notebook runs in JupyterLite (Python 3.14 inside your browser) and on your JupyterHub server (Python 3.13). It needs no internet access and runs from top to bottom in a few seconds.

### How to use this notebook

* Run the cells in order, from the top. **Run → Run All Cells** works too.
* Each implementation cell is followed by a test cell of plain `assert` statements. A test cell that reaches its final `print` has passed; a failing assertion stops the cell with a message that says what went wrong.
* The exercises use a small helper called `check`, defined in the next cell. It prints `not implemented yet` for an exercise you have not attempted, so the whole notebook runs before you start.
* Reference solutions are at the very end. Try each exercise first.

In [ ]:
import copy
import math
import numbers


def _matches(result, expected, rel_tol, abs_tol):
    """True when result equals expected: exactly for integers and booleans,
    within a tolerance for floats, and entry by entry for lists and tuples."""
    if expected is None or isinstance(expected, bool):
        return result is expected
    if isinstance(expected, numbers.Real):
        if isinstance(result, bool) or not isinstance(result, numbers.Real):
            return False
        if isinstance(expected, int):
            return result == expected
        return math.isclose(result, expected, rel_tol=rel_tol, abs_tol=abs_tol)
    if isinstance(expected, (list, tuple)):
        return (
            isinstance(result, (list, tuple))
            and len(result) == len(expected)
            and all(_matches(r, e, rel_tol, abs_tol) for r, e in zip(result, expected))
        )
    return result == expected


def check(name, function, cases, *, rel_tol=1e-9, abs_tol=1e-9, strict=False):
    """Test an exercise function against a list of (arguments, expected) cases.

    arguments is a tuple of positional arguments, for example ([1, 2, 3],) for a
    function that takes one list. expected is the value the function should
    return, or an exception class such as ValueError that it should raise.
    Integers must match exactly; floats are compared with math.isclose.

    Prints "<name>: not implemented yet" while the function still raises
    NotImplementedError, prints a pass line when every case is right, and raises
    AssertionError with the failing call when a case is wrong. strict=True
    (used in the Solutions section) treats an unimplemented function as a failure.
    """
    for arguments, expected in cases:
        call = f"{name}({', '.join(repr(argument) for argument in arguments)})"
        wants_error = isinstance(expected, type) and issubclass(expected, Exception)
        # Work on copies of list and dict arguments so that a function which
        # changes its input is caught instead of silently corrupting later cases.
        supplied = tuple(
            copy.deepcopy(argument) if isinstance(argument, (list, dict)) else argument
            for argument in arguments
        )
        try:
            result = function(*supplied)
        except NotImplementedError:
            if strict:
                raise AssertionError(f"{call} raised NotImplementedError; finish the function first")
            print(f"{name}: not implemented yet")
            return
        except Exception as error:
            if wants_error and isinstance(error, expected):
                continue
            wanted = f"it to raise {expected.__name__}" if wants_error else repr(expected)
            raise AssertionError(
                f"{call} raised {type(error).__name__}: {error}; expected {wanted}"
            ) from error
        if wants_error:
            raise AssertionError(f"{call} returned {result!r}; expected it to raise {expected.__name__}")
        for before, after in zip(arguments, supplied):
            if isinstance(before, (list, dict)) and before != after:
                raise AssertionError(
                    f"{call} changed its input {before!r} into {after!r}; "
                    "return a new list instead of modifying the argument"
                )
        if not _matches(result, expected, rel_tol, abs_tol):
            raise AssertionError(f"{call} returned {result!r}; expected {expected!r}")
    summary = "the test" if len(cases) == 1 else f"all {len(cases)} tests"
    print(f"{name}: {summary} passed")



def raises_value_error(function, *arguments):
    """True when function(*arguments) raises ValueError (used by the tests below)."""
    try:
        function(*arguments)
    except ValueError:
        return True
    return False

The next cell shows the three things `check` can report: a pass, an exercise that has not been started, and a wrong answer. The wrong answer is caught here only so that the notebook keeps running; in the exercises, the `AssertionError` stops the cell and shows you the failing call.

In [ ]:
def square(x):
    return x * x


def cube(x):
    # TODO: an exercise that has not been attempted yet
    raise NotImplementedError("Return x * x * x")


def wrong_square(x):
    return 2 * x


check("square", square, [((3,), 9), ((-1.5,), 2.25)])
check("cube", cube, [((2,), 8)])
try:
    check("wrong_square", wrong_square, [((2,), 4), ((3,), 9)])
except AssertionError as error:
    print("AssertionError:", error)

**Expected behaviour.** Three lines appear: `square: all 2 tests passed`, `cube: not implemented yet` and `AssertionError: wrong_square(3) returned 6; expected 9`. The wrong function passes its first case by coincidence, because $2 \times 2 = 2^2$, and fails on the second. One lucky test proves very little, so every exercise is checked on several cases.

## Theory

### Integers and floating-point numbers

Python has two everyday kinds of number.

* An **`int`** is an exact integer of any size. Python stores $2^{100}$ or $30!$ digit for digit, so integer addition, subtraction and multiplication never round.
* A **`float`** is an IEEE 754 double-precision binary number. It stores a sign, a 53-bit integer significand $m$ and an exponent $e$:

$$x = \pm\, m \times 2^{e}, \qquad 2^{52} \le m < 2^{53}.$$

Only finitely many such numbers exist, so most real numbers, including $0.1$, are rounded to the nearest one that can be stored. The gap between $1$ and the next float is the **machine epsilon** $\varepsilon = 2^{-52} \approx 2.2 \times 10^{-16}$. Rounding to the nearest float changes a value by a relative amount of at most $u = \varepsilon / 2 = 2^{-53}$, which is why a float carries about 15 to 17 significant decimal digits. Floats also have a limited range: the largest is about $1.8 \times 10^{308}$.

### Expressions

Python follows the usual precedence rules, with a few details worth learning by heart.

| Expression | Value | Reason |
|---|---|---|
| `7 / 2` | `3.5` | `/` is true division and always returns a float |
| `6 / 3` | `2.0` | still a float, even when the division is exact |
| `7 // 2` | `3` | floor division rounds down, towards $-\infty$ |
| `-7 // 2` | `-4` | so $-3.5$ becomes $-4$, not $-3$ |
| `-7 % 2` | `1` | the remainder satisfies `(a // b) * b + a % b == a` |
| `2 ** -1` | `0.5` | a negative exponent produces a float |
| `-3 ** 2` | `-9` | `**` binds more tightly than the minus sign, so this is $-(3^2)$ |

### Functions as mathematical notation

A mathematical function assigns one output to each input, for example $f(x) = 3x^2 - 2x + 1$. A Python function written with `def` plays the same role: its parameters are the inputs and `return` gives the output. Python needs every multiplication written out, so $3x^2$ becomes `3 * x ** 2`.

* A **piecewise** function uses `if`. The rectified linear unit used in neural networks is $\operatorname{ReLU}(x) = \max(0, x)$.
* The **logistic sigmoid** $\sigma(x) = \dfrac{1}{1 + e^{-x}}$ squashes any real number into the interval $(0, 1)$. For a large negative $x$ the term $e^{-x}$ is too large for a float, so a careful implementation uses the equivalent form $\sigma(x) = \dfrac{e^{x}}{1 + e^{x}}$ when $x < 0$.
* **Composition** $(g \circ f)(x) = g(f(x))$ applies $f$ first and then $g$. Python functions are values, so a function can take other functions as arguments and return a new one.

### Sums, products and recursion

Summation and product notation are loops with an accumulator:

$$\sum_{i=1}^{n} i = 1 + 2 + \cdots + n = \frac{n(n+1)}{2}, \qquad n! = \prod_{i=1}^{n} i = 1 \times 2 \times \cdots \times n.$$

A sum starts its accumulator at $0$ and a product starts at $1$. That is why an **empty sum** is $0$ and an **empty product** is $1$, so $0! = 1$. The factorial also has a recursive definition, with a **base case** that stops the recursion and a **recursive case** that reduces the problem to a smaller one:

$$0! = 1, \qquad n! = n \times (n-1)! \quad \text{for } n \ge 1.$$

Python limits how deeply functions may call themselves (usually to about 1000 nested calls), so a loop is the safer choice for long computations.

### Vectors and matrices as lists

A vector $\mathbf{x} = (x_1, x_2, \ldots, x_n)$ can be stored as a list, and a matrix as a **list of rows**. For example,

$$A = \begin{pmatrix} 1 & 2 & 3 \\ 4 & 5 & 6 \end{pmatrix}$$

becomes `A = [[1, 2, 3], [4, 5, 6]]`. This $A$ has 2 rows and 3 columns, so its **shape** is $2 \times 3$. Mathematics counts from 1 but Python counts from 0, so the entry $a_{ij}$ in row $i$ and column $j$ is `A[i - 1][j - 1]`; for example, $a_{12} = 2$ is `A[0][1]`. Every row of a matrix must have the same length. A list whose rows differ in length is called **ragged**, and it is not a matrix.

List comprehensions read like set-builder notation: $\{x^2 : x \in S\}$ becomes `[x ** 2 for x in S]`.

### Classes and operators

A class bundles data with the operations that act on it. Special methods, whose names start and end with two underscores, connect a class to Python's operators: `u + v` calls `u.__add__(v)`, and `repr(v)` calls `v.__repr__()`. A 2-D vector class can therefore follow the mathematics directly:

$$\mathbf{u} + \mathbf{v} = (u_1 + v_1,\ u_2 + v_2), \qquad c\,\mathbf{v} = (c\,v_1,\ c\,v_2), \qquad \|\mathbf{v}\| = \sqrt{v_1^2 + v_2^2}.$$

### Floating-point precision

Because each float is rounded, the result of each arithmetic operation is rounded too. In the standard model of floating-point arithmetic, for each operation $\circ$ among $+, -, \times, \div$,

$$\operatorname{fl}(a \circ b) = (a \circ b)(1 + \delta), \qquad |\delta| \le u = 2^{-53}.$$

A single rounding is tiny, but there are three common ways for rounding errors to become visible.

* **Representation error.** $0.1$, $0.2$ and $0.3$ have no exact binary form, so `0.1 + 0.2 == 0.3` is `False`. Compare floats with a tolerance instead. `math.isclose(a, b)` accepts $|a - b| \le \max(r \cdot \max(|a|, |b|),\ t)$, where $r$ is `rel_tol` (by default $10^{-9}$) and $t$ is `abs_tol` (by default $0$). A purely relative tolerance never accepts a non-zero value when the other side is exactly $0$, so pass `abs_tol` when you compare with zero.
* **Accumulation error.** Adding many numbers one at a time rounds after every step, and the errors can build up. `math.fsum` keeps track of the exact running total and rounds only once, at the end.
* **Catastrophic cancellation.** Subtracting two nearly equal numbers cancels their leading digits. The subtraction itself is exact, but the rounding errors that the two numbers already carried become a large *relative* error in the small difference. The cure is to rearrange the formula. For example, the population variance is better computed with the two-pass formula $\frac{1}{n}\sum_{i}(x_i - \bar{x})^2$ than with the algebraically equal shortcut $\frac{1}{n}\sum_{i} x_i^2 - \bar{x}^2$.

## Step-by-step implementation

### Step 1: integers, floats and expressions

Start by looking at the two kinds of number directly. The cell below prints a few values; match each line with the theory above.

In [ ]:
import sys

print("2 ** 100 =", 2 ** 100)
print("30! =", math.factorial(30), "has", len(str(math.factorial(30))), "digits, all of them exact")
print("Largest float:  ", sys.float_info.max)
print("Machine epsilon:", sys.float_info.epsilon)
print()
print("2.0 ** 53 + 1 == 2.0 ** 53 ->", 2.0 ** 53 + 1 == 2.0 ** 53, "(floats run out of digits)")
print("2 ** 53 + 1 == 2 ** 53     ->", 2 ** 53 + 1 == 2 ** 53, "(integers never do)")
print()
print("7 / 2 =", 7 / 2, "  6 / 3 =", 6 / 3, "  7 // 2 =", 7 // 2, "  -7 // 2 =", -7 // 2, "  -7 % 2 =", -7 % 2)
print("-3 ** 2 =", -3 ** 2, "  (-3) ** 2 =", (-3) ** 2, "  2 ** -1 =", 2 ** -1)

Now compute the machine epsilon yourself instead of reading it from `sys.float_info`. Start from $\varepsilon = 1$ and keep halving it for as long as $1 + \varepsilon/2$ is still different from $1$. The loop stops at the smallest power of two that still changes $1$ when it is added to it.

In [ ]:
def machine_epsilon():
    """Return the gap between 1.0 and the next larger float, found by halving."""
    epsilon = 1.0
    while 1.0 + epsilon / 2 != 1.0:
        epsilon /= 2
    return epsilon

In [ ]:
# Tests for Step 1
eps = machine_epsilon()
assert eps == 2.0 ** -52, f"expected 2 ** -52, got {eps!r}"
assert eps == sys.float_info.epsilon, "the halving loop should agree with sys.float_info.epsilon"
assert 1.0 + eps != 1.0, "adding epsilon to 1.0 must change it"
assert 1.0 + eps / 2 == 1.0, "adding half of epsilon to 1.0 rounds back to 1.0"
assert 2.0 ** 53 + 1 == 2.0 ** 53, "2 ** 53 + 1 cannot be stored as a float"
assert 2 ** 53 + 1 != 2 ** 53, "integers stay exact"
assert 6 / 3 == 2.0 and isinstance(6 / 3, float), "true division always returns a float"
assert -7 // 2 == -4, "floor division rounds towards minus infinity"
assert -3 ** 2 == -9 and (-3) ** 2 == 9, "** binds more tightly than unary minus"
for a in range(-10, 11):
    for b in (-3, -2, 2, 3):
        assert (a // b) * b + a % b == a, f"the division identity failed for a={a}, b={b}"
print("Step 1 tests passed: machine epsilon =", eps)

**Expected behaviour.** The loop halves $\varepsilon$ 52 times and returns `2.220446049250313e-16`, which is exactly $2^{-52}$ and the same value as `sys.float_info.epsilon`. The integer $2^{53} + 1$ is stored exactly, but as a float it rounds back to $2^{53}$. The floor-division identity holds for every pair tested, including negative numbers.

### Step 2: functions as mathematical notation

Each function below is a direct translation of a formula from the theory. `sigmoid` switches to the stable form for negative inputs; `naive_sigmoid` is there only to show what goes wrong without it.

In [ ]:
def f(x):
    """f(x) = 3x^2 - 2x + 1."""
    return 3 * x ** 2 - 2 * x + 1


def relu(x):
    """ReLU(x) = max(0, x)."""
    return max(0.0, x)


def naive_sigmoid(x):
    """1 / (1 + e^(-x)), exactly as written. It overflows for large negative x."""
    return 1.0 / (1.0 + math.exp(-x))


def sigmoid(x):
    """The logistic function, arranged so that math.exp never overflows."""
    if x >= 0:
        return 1.0 / (1.0 + math.exp(-x))
    z = math.exp(x)  # x < 0, so 0 < z < 1
    return z / (1.0 + z)


def compose(outer, inner):
    """Return the function x -> outer(inner(x)), written outer ∘ inner."""
    def composed(x):
        return outer(inner(x))
    return composed

In [ ]:
# Tests for Step 2
assert f(0) == 1 and f(2) == 9 and f(-1) == 6, "f gives 1, 9 and 6 at x = 0, 2 and -1"
assert relu(-2.5) == 0.0 and relu(3) == 3, "ReLU zeroes negative inputs and keeps positive ones"
assert math.isclose(sigmoid(0), 0.5), "sigmoid(0) is one half"
for x in (-30.0, -2.0, -0.5, 0.5, 2.0, 30.0):
    assert math.isclose(sigmoid(x) + sigmoid(-x), 1.0), f"sigmoid(-x) should equal 1 - sigmoid(x) at x = {x}"
    assert math.isclose(sigmoid(x), naive_sigmoid(x)), f"both forms should agree where neither overflows (x = {x})"
assert 0.0 <= sigmoid(-1000) < 1e-300 and math.isclose(sigmoid(1000), 1.0), "extreme inputs saturate instead of overflowing"
try:
    naive_sigmoid(-1000)
except OverflowError:
    pass
else:
    raise AssertionError("naive_sigmoid(-1000) was expected to raise OverflowError")
relu_then_f = compose(f, relu)
f_then_relu = compose(relu, f)
assert relu_then_f(-1) == 1, "f(relu(-1)) = f(0) = 1"
assert f_then_relu(-1) == 6, "relu(f(-1)) = relu(6) = 6"
print("Step 2 tests passed: sigmoid(2) =", sigmoid(2))

**Expected behaviour.** $f$ gives $1$, $9$ and $6$ at $x = 0, 2, -1$. The sigmoid satisfies $\sigma(-x) = 1 - \sigma(x)$, returns exactly $0.5$ at zero, and saturates to $0.0$ and $1.0$ at $\pm 1000$. The naive version raises `OverflowError` at $x = -1000$, because $e^{1000}$ is far beyond the largest float: Python's `math` functions raise an error rather than return infinity. Composition order matters, since $(f \circ \operatorname{ReLU})(-1) = 1$ but $(\operatorname{ReLU} \circ f)(-1) = 6$.

### Step 3: loops and recursion

`sum_to` and `sum_of_squares` turn $\sum$ into a loop with an accumulator that starts at $0$. The two factorial functions compute the same product, once with a loop and once recursively. The tests compare all four with the closed-form formulas and with `math.factorial`.

In [ ]:
def sum_to(n):
    """1 + 2 + ... + n, using a loop. Returns 0 (the empty sum) when n is 0."""
    total = 0
    for i in range(1, n + 1):  # range stops before its end, hence n + 1
        total += i
    return total


def sum_of_squares(n):
    """1**2 + 2**2 + ... + n**2, using a loop."""
    total = 0
    for i in range(1, n + 1):
        total += i * i
    return total


def factorial_iterative(n):
    """n! as a running product that starts at 1, the empty product."""
    if n < 0:
        raise ValueError(f"the factorial is undefined for negative n, got {n}")
    product = 1
    for i in range(2, n + 1):
        product *= i
    return product


def factorial_recursive(n):
    """n! from the definition 0! = 1 and n! = n * (n - 1)!."""
    if n < 0:
        raise ValueError(f"the factorial is undefined for negative n, got {n}")
    if n == 0:  # base case: stops the recursion
        return 1
    return n * factorial_recursive(n - 1)  # recursive case: a smaller problem

In [ ]:
# Tests for Step 3
for n in range(60):
    assert sum_to(n) == n * (n + 1) // 2, f"sum_to({n}) disagrees with n(n + 1)/2"
    assert sum_of_squares(n) == n * (n + 1) * (2 * n + 1) // 6, f"sum_of_squares({n}) disagrees with the closed form"
    assert factorial_iterative(n) == math.factorial(n), f"factorial_iterative({n}) is wrong"
    assert factorial_recursive(n) == math.factorial(n), f"factorial_recursive({n}) is wrong"
assert sum_to(0) == 0 and factorial_iterative(0) == 1, "the empty sum is 0 and the empty product is 1"
assert raises_value_error(factorial_iterative, -1), "factorial_iterative(-1) should raise ValueError"
assert raises_value_error(factorial_recursive, -1), "factorial_recursive(-1) should raise ValueError"
print("Step 3 tests passed: sum_to(100) =", sum_to(100), "and 20! =", factorial_iterative(20))

**Expected behaviour.** Every value for $n = 0, 1, \ldots, 59$ matches the closed forms, and both factorials agree with `math.factorial`. `sum_to(100)` is $5050$, the total the young Gauss is said to have found in his head, and $20! = 2432902008176640000$ is exact because Python integers never round. A negative argument raises `ValueError` with a message, instead of recursing until Python gives up.

### Step 4: lists and nested lists

`shape` returns the number of rows and columns of a list-of-rows matrix and rejects anything that is not rectangular. `zeros` builds a matrix of zeros with a **separate** list for each row, and `column` collects one column. The demonstration after them shows why separate rows matter.

In [ ]:
def shape(matrix):
    """Return (rows, columns) of a matrix stored as a list of rows.

    Raises ValueError for an empty matrix, an empty row or ragged rows,
    because none of these is a valid m x n matrix.
    """
    if len(matrix) == 0:
        raise ValueError("a matrix needs at least one row")
    columns = len(matrix[0])
    if columns == 0:
        raise ValueError("a matrix needs at least one column")
    for i, row in enumerate(matrix):
        if len(row) != columns:
            raise ValueError(f"row {i} has {len(row)} entries but row 0 has {columns}: the rows are ragged")
    return len(matrix), columns


def zeros(rows, columns):
    """A rows x columns matrix of zeros, with a new list for every row."""
    return [[0.0] * columns for _ in range(rows)]


def column(matrix, j):
    """Column j (counting from 0) of a matrix, as a list."""
    shape(matrix)  # validate before reading
    return [row[j] for row in matrix]

In [ ]:
A = [[1, 2, 3],
     [4, 5, 6]]
print("A has shape", shape(A))
print("a_12 (row 1, column 2) is A[0][1] =", A[0][1])
print("The column with index 2:", column(A, 2))
print("{x^2 : x in 1, ..., 5} ->", [x ** 2 for x in range(1, 6)])

shared = [[0] * 3] * 2  # the outer * copies a reference to ONE row list
shared[0][0] = 99
print("Built with [[0] * 3] * 2, after setting one entry:", shared)

separate = zeros(2, 3)  # a new list for every row
separate[0][0] = 99
print("Built with zeros(2, 3), after setting one entry: ", separate)

In [ ]:
# Tests for Step 4
assert shape([[1, 2, 3], [4, 5, 6]]) == (2, 3), "a 2 x 3 matrix has 2 rows and 3 columns"
assert shape([[7]]) == (1, 1), "a single entry is a 1 x 1 matrix"
assert shape([[1], [2], [3]]) == (3, 1), "a column vector has shape n x 1"
assert raises_value_error(shape, []), "an empty list is not a matrix"
assert raises_value_error(shape, [[]]), "a matrix with an empty row has no columns"
assert raises_value_error(shape, [[1, 2], [3]]), "a ragged list is not a matrix"
assert column(A, 1) == [2, 5], "column 1 of A is [2, 5]"
assert zeros(2, 2) == [[0.0, 0.0], [0.0, 0.0]], "zeros(2, 2) is a 2 x 2 matrix of zeros"
assert separate[1][0] == 0.0, "rows made by zeros() must be independent lists"
assert shared[1][0] == 99, "rows made by [[0] * 3] * 2 are one and the same list"
print("Step 4 tests passed")

**Expected behaviour.** `A` has shape `(2, 3)` and $a_{12}$ is `A[0][1] == 2`. An empty list, a matrix with an empty row and a ragged list all raise `ValueError`, each with a message that names the problem. The demonstration shows the classic aliasing trap: `[[0] * 3] * 2` creates two references to **one** row, so setting a single entry changes both rows to `[99, 0, 0]`. `zeros` builds every row separately, so only the first row changes.

### Step 5: a class for 2-D vectors

`Vector2D` stores two components and defines the special methods that give it mathematical notation. Each operator method returns `NotImplemented` for an operand it does not understand; Python then gives the other operand a chance and, if that fails too, raises a clear `TypeError`. `__rmul__` handles `3 * v`: Python first asks the integer `3` how to multiply itself by a vector, the integer answers `NotImplemented`, and Python then asks the vector.

In [ ]:
class Vector2D:
    """A vector (x, y) in the plane that supports +, -, scalar * and repr()."""

    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __repr__(self):
        return f"Vector2D({self.x!r}, {self.y!r})"

    def __add__(self, other):
        if not isinstance(other, Vector2D):
            return NotImplemented  # lets Python raise a clear TypeError
        return Vector2D(self.x + other.x, self.y + other.y)

    def __sub__(self, other):
        if not isinstance(other, Vector2D):
            return NotImplemented
        return Vector2D(self.x - other.x, self.y - other.y)

    def __mul__(self, scalar):
        if not isinstance(scalar, (int, float)):
            return NotImplemented
        return Vector2D(scalar * self.x, scalar * self.y)

    def __rmul__(self, scalar):
        return self.__mul__(scalar)  # makes 3 * v work as well as v * 3

    def dot(self, other):
        return self.x * other.x + self.y * other.y

    def norm(self):
        return math.sqrt(self.dot(self))

In [ ]:
# Tests for Step 5
u = Vector2D(1, 2)
v = Vector2D(3, -1)
w = u + v
assert (w.x, w.y) == (4, 1), "u + v adds component by component"
d = u - v
assert (d.x, d.y) == (-2, 3), "u - v subtracts component by component"
s = 3 * u
assert (s.x, s.y) == (3, 6), "3 * u scales both components"
t = u * 0.5
assert math.isclose(t.x, 0.5) and math.isclose(t.y, 1.0), "u * 0.5 scales too"
assert u.dot(v) == 1, "(1)(3) + (2)(-1) = 1"
assert math.isclose(Vector2D(3, 4).norm(), 5.0), "the 3-4-5 right triangle"
assert repr(u) == "Vector2D(1, 2)", "repr shows how to rebuild the object"
try:
    u + 5
except TypeError:
    pass
else:
    raise AssertionError("adding a plain number to a vector should raise TypeError")
print(u, "+", v, "=", w)

**Expected behaviour.** The cell prints `Vector2D(1, 2) + Vector2D(3, -1) = Vector2D(4, 1)`. Addition and subtraction work component by component, `3 * u` and `u * 0.5` both scale, and the 3–4–5 triangle gives a norm of `5.0`. Adding a plain number to a vector raises `TypeError`, which is far better than silently returning a wrong answer.

### Step 6: floating-point precision

First look at what is really stored. `fractions.Fraction` and `decimal.Decimal` both reveal the exact value of a float. Then look at representation, accumulation and cancellation errors one at a time.

In [ ]:
from decimal import Decimal
from fractions import Fraction

print("0.1 + 0.2 =", 0.1 + 0.2)
print("0.1 is stored as", Fraction(0.1), "=", Decimal(0.1))
print("math.isclose(0.1 + 0.2, 0.3) ->", math.isclose(0.1 + 0.2, 0.3))
print("math.isclose(1e-20, 0.0) ->", math.isclose(1e-20, 0.0),
      "| with abs_tol=1e-12 ->", math.isclose(1e-20, 0.0, abs_tol=1e-12))
print()
print("(1e16 + 1.0) - 1e16 =", (1e16 + 1.0) - 1e16, "but (1e16 - 1e16) + 1.0 =", (1e16 - 1e16) + 1.0)
difference = (1 + 1e-15) - 1
print("(1 + 1e-15) - 1 =", difference, "a relative error of", abs(difference - 1e-15) / 1e-15)
print("math.log(1 + 1e-15) =", math.log(1 + 1e-15), "but math.log1p(1e-15) =", math.log1p(1e-15))

Next, two small functions. `loop_sum` adds numbers one at a time, as a hand-written loop does in any language. `shortcut_variance` computes the population variance with the shortcut formula $\frac{1}{n}\sum_i x_i^2 - \bar{x}^2$, which is algebraically correct but numerically fragile. Both exist to *show* errors; neither is a function to copy.

In [ ]:
import statistics


def loop_sum(values):
    """Add the values one at a time, rounding after every addition."""
    total = 0.0
    for value in values:
        total += value
    return total


def shortcut_variance(values):
    """Population variance as mean(x^2) - mean(x)^2. Prone to cancellation."""
    n = len(values)
    mean_of_squares = loop_sum([x * x for x in values]) / n
    mean_value = loop_sum(values) / n
    return mean_of_squares - mean_value * mean_value


for count in (10, 1_000, 1_000_000):
    values = [0.1] * count
    print(f"{count:>9,} additions of 0.1: loop {loop_sum(values)!r:<20} fsum {math.fsum(values)!r}")

shifted = [1e9 + 4, 1e9 + 7, 1e9 + 13, 1e9 + 16]
print()
print("shortcut_variance([4, 7, 13, 16]) =", shortcut_variance([4, 7, 13, 16]))
print("shortcut_variance(shifted)        =", shortcut_variance(shifted))
print("statistics.pvariance(shifted)     =", statistics.pvariance(shifted))

In [ ]:
# Tests for Step 6
assert 0.1 + 0.2 != 0.3, "0.1 + 0.2 is not exactly 0.3 in binary floating point"
assert math.isclose(0.1 + 0.2, 0.3), "but it is within the default relative tolerance"
assert not math.isclose(1e-20, 0.0), "a relative tolerance alone rejects any non-zero value compared with 0.0"
assert math.isclose(1e-20, 0.0, abs_tol=1e-12), "abs_tol makes comparisons with zero work"
assert loop_sum([0.1] * 10) == 0.9999999999999999, "ten rounded additions of 0.1 fall just short of 1"
assert math.fsum([0.1] * 10) == 1.0, "fsum rounds once, at the end"
assert abs(loop_sum([0.1] * 1_000_000) - 100_000) > 1e-7, "a million additions drift visibly"
assert math.fsum([0.1] * 1_000_000) == 100_000.0, "fsum does not drift"
assert (1e16 + 1.0) - 1e16 == 0.0, "1.0 is absorbed, because floats near 1e16 are 2 apart"
assert (1e16 - 1e16) + 1.0 == 1.0, "in another order the 1.0 survives: float addition is not associative"
assert math.isclose(shortcut_variance([4, 7, 13, 16]), 22.5), "the shortcut is fine for small, well-scaled numbers"
assert abs(shortcut_variance(shifted) - 22.5) > 100, "cancellation ruins the shortcut when the mean is large"
assert math.isclose(statistics.pvariance(shifted), 22.5), "a careful method still finds the true variance"
print("Step 6 tests passed")

**Expected behaviour.** `0.1` is stored as $3602879701896397 / 2^{55}$, slightly more than one tenth, so `0.1 + 0.2` prints `0.30000000000000004`. `math.isclose` accepts that result, but it rejects `1e-20` against `0.0` until you supply `abs_tol`. Adding `0.1` in a loop drifts: ten additions give `0.9999999999999999`, and a million give an error of about $1.3 \times 10^{-6}$, while `math.fsum` returns the correctly rounded totals. Adding `1.0` to `1e16` changes nothing, because neighbouring floats of that size are 2 apart. The subtraction $(1 + 10^{-15}) - 1$ is exact in itself, yet it exposes the rounding of $1 + 10^{-15}$ and gives an 11% relative error; `math.log1p` avoids the same trap for $\log(1 + x)$. Finally, the shortcut variance is right for `[4, 7, 13, 16]` but returns `-128.0`, a negative variance, once $10^9$ is added to every value. The true variance is still $22.5$.

Python's built-in `sum` has used a compensated algorithm for floats since Python 3.12, so `sum([0.1] * 10)` returns `1.0`. Hand-written loops, older Python versions and many other languages behave like `loop_sum`, so reaching for `math.fsum` is still a good habit.

### Optional comparison: NumPy's float types

*Optional: you can skip this cell, and nothing later depends on it.* Machine-learning libraries often store numbers as 32-bit floats to save memory and time, and NumPy makes the difference easy to see. If NumPy is not installed, the cell prints a note instead of failing.

In [ ]:
try:
    import numpy as np
except ImportError:
    print("NumPy is not available here; skipping this optional comparison.")
else:
    print("float64 machine epsilon:", np.finfo(np.float64).eps, "(the same as Python's float)")
    print("float32 machine epsilon:", np.finfo(np.float32).eps)
    print("0.1 stored as a float32 is", float(np.float32(0.1)))
    with np.errstate(over="ignore"):
        print("np.exp(1000.0) returns", np.exp(np.float64(1000.0)), "where math.exp(1000) raises OverflowError")

**Expected behaviour.** A 32-bit float has a machine epsilon of about $1.2 \times 10^{-7}$, so it carries only about 7 significant digits, and $0.1$ becomes `0.10000000149011612`. NumPy returns `inf` for an overflowing exponential instead of raising an error, which is faster but easier to miss.

## Exercises

Each exercise gives you a function with a docstring, a `TODO` comment and a `raise NotImplementedError(...)` line. Replace that line with your own code, run the cell, then run the `check` cell below it. Until you do, `check` prints `not implemented yet`, so the notebook still runs from top to bottom.

You may use anything from the standard library and any function defined in the implementation steps above. The reference solutions are at the end of the notebook; try each exercise before you look.

### Exercise 1: mathematical functions

**(a) Evaluate a polynomial.** Write `polynomial(coefficients, x)` for

$$p(x) = c_0 + c_1 x + c_2 x^2 + \cdots + c_n x^n,$$

where `coefficients` lists $c_0, c_1, \ldots, c_n$ from the constant term upwards. An empty list is the zero polynomial, so return `0`. *Hint:* Horner's rule, $p(x) = c_0 + x\,(c_1 + x\,(c_2 + \cdots + x\,c_n))$, needs only one multiplication per coefficient. Start with `result = 0` and work through the coefficients from the **last** to the first.

**(b) Count combinations.** Write `n_choose_k(n, k)` for the binomial coefficient

$$\binom{n}{k} = \frac{n!}{k!\,(n-k)!} = \prod_{i=1}^{k} \frac{n - k + i}{i},$$

the number of ways to choose $k$ items from $n$. Return `0` when $k > n$, and raise `ValueError` when $n$ or $k$ is negative; this matches `math.comb`. Use integer arithmetic only, so that large results stay exact: multiply by $n - k + i$ first and then divide by $i$ with `//` at every step, and each intermediate result is itself a whole number.

In [ ]:
def polynomial(coefficients, x):
    """Return c0 + c1*x + c2*x**2 + ... for coefficients [c0, c1, c2, ...]."""
    # TODO: use Horner's rule, looping over the coefficients from last to first.
    raise NotImplementedError("Exercise 1a: implement polynomial")


def n_choose_k(n, k):
    """Return the binomial coefficient C(n, k) as an exact integer."""
    # TODO: raise ValueError for a negative n or k, return 0 when k > n,
    #       otherwise build the product one factor at a time with integers.
    raise NotImplementedError("Exercise 1b: implement n_choose_k")

In [ ]:
POLYNOMIAL_CASES = [
    (([1], 5), 1),
    (([1, -2, 3], 2), 9),  # the f(x) from Step 2: 1 - 4 + 12
    (([1, -2, 3], -1), 6),
    (([0, 0, 0, 1], 1.5), 3.375),  # x cubed
    (([2, 0.5], 0.1), 2.05),
    (([], 7), 0),
]
N_CHOOSE_K_CASES = [
    ((5, 2), 10),
    ((10, 0), 1),
    ((10, 10), 1),
    ((52, 5), 2_598_960),  # five-card hands from a 52-card deck
    ((100, 50), math.comb(100, 50)),  # 30 digits, so it must be exact
    ((3, 5), 0),
    ((-1, 2), ValueError),
    ((5, -1), ValueError),
]
check("polynomial", polynomial, POLYNOMIAL_CASES)
check("n_choose_k", n_choose_k, N_CHOOSE_K_CASES)

### Exercise 2: sums, averages and variances

Write three functions. Each should raise `ValueError` for an empty input, rather than fail with a `ZeroDivisionError`.

* `mean(values)` returns $\bar{x} = \frac{1}{n}\sum_{i=1}^{n} x_i$. Use `math.fsum` for the sum.
* `variance(values)` returns the **population** variance $\frac{1}{n}\sum_{i=1}^{n} (x_i - \bar{x})^2$, dividing by $n$. Use the two-pass method: find the mean first, then average the squared deviations from it. One test adds $10^9$ to every value, which defeats the shortcut formula from Step 6. Lesson 3 explains when to divide by $n - 1$ instead.
* `mean_squared_error(predictions, targets)` returns $\frac{1}{n}\sum_{i=1}^{n} (\hat{y}_i - y_i)^2$, the most common loss for regression. Raise `ValueError` if the two lists are empty or have different lengths.

Your `variance` may call your `mean`.

In [ ]:
def mean(values):
    """Arithmetic mean of a non-empty list of numbers."""
    # TODO: raise ValueError for an empty list, then divide math.fsum(values) by n.
    raise NotImplementedError("Exercise 2a: implement mean")


def variance(values):
    """Population variance (divide by n), computed with the two-pass method."""
    # TODO: raise ValueError for an empty list, find the mean,
    #       then average the squared deviations from it.
    raise NotImplementedError("Exercise 2b: implement variance")


def mean_squared_error(predictions, targets):
    """Average of (prediction - target) ** 2 over two equally long, non-empty lists."""
    # TODO: validate the lengths, then average the squared differences.
    raise NotImplementedError("Exercise 2c: implement mean_squared_error")

In [ ]:
SHIFTED = [1e9 + 4, 1e9 + 7, 1e9 + 13, 1e9 + 16]
MEAN_CASES = [
    (([1, 2, 3, 4],), 2.5),
    (([5],), 5.0),
    (([0.1] * 10,), 0.1),
    (([-3, 3],), 0.0),
    ((SHIFTED,), 1e9 + 10),
    (([],), ValueError),
]
VARIANCE_CASES = [
    (([4, 7, 13, 16],), 22.5),
    (([2, 4, 4, 4, 5, 5, 7, 9],), 4.0),
    (([5, 5, 5],), 0.0),
    ((SHIFTED,), 22.5),  # the shortcut formula fails this case
    (([],), ValueError),
]
MSE_CASES = [
    (([1, 2, 3], [1, 2, 3]), 0.0),
    (([2.5, 0.0, 2.0], [3.0, -0.5, 2.0]), 0.5 / 3),
    (([1], [3]), 4.0),
    (([1, 2], [1]), ValueError),
    (([], []), ValueError),
]
check("mean", mean, MEAN_CASES)
check("variance", variance, VARIANCE_CASES)
check("mean_squared_error", mean_squared_error, MSE_CASES)

### Exercise 3: matrices as nested lists

* `reshape(values, rows, columns)` turns a flat list into a `rows` × `columns` matrix, filling it **row by row**: `reshape([1, 2, 3, 4, 5, 6], 2, 3)` is `[[1, 2, 3], [4, 5, 6]]`. Raise `ValueError` unless `rows` and `columns` are at least 1 and `len(values) == rows * columns`.
* `flatten(matrix)` does the reverse and returns the entries row by row in a single list. Call `shape(matrix)` first, so that ragged input raises `ValueError`.
* `diagonal_matrix(diagonal)` returns the square matrix with `diagonal` on its main diagonal and zeros everywhere else, for example

$$\operatorname{diag}(1, 2, 3) = \begin{pmatrix} 1 & 0 & 0 \\ 0 & 2 & 0 \\ 0 & 0 & 3 \end{pmatrix}.$$

Raise `ValueError` for an empty list, and watch out for the aliasing trap from Step 4.

For every valid matrix `M`, `reshape(flatten(M), *shape(M))` should give back `M`.

In [ ]:
def reshape(values, rows, columns):
    """Arrange a flat list into a rows x columns list of rows, filling row by row."""
    # TODO: validate the sizes, then slice the values into consecutive rows.
    raise NotImplementedError("Exercise 3a: implement reshape")


def flatten(matrix):
    """Return every entry of a rectangular matrix, row by row, in one list."""
    # TODO: validate with shape(matrix), then collect the entries.
    raise NotImplementedError("Exercise 3b: implement flatten")


def diagonal_matrix(diagonal):
    """Square matrix with the given diagonal entries and zeros elsewhere."""
    # TODO: build a new list for every row (see Step 4), then fill in the diagonal.
    raise NotImplementedError("Exercise 3c: implement diagonal_matrix")

In [ ]:
RESHAPE_CASES = [
    (([1, 2, 3, 4, 5, 6], 2, 3), [[1, 2, 3], [4, 5, 6]]),
    (([1, 2, 3, 4, 5, 6], 3, 2), [[1, 2], [3, 4], [5, 6]]),
    (([1, 2, 3], 1, 3), [[1, 2, 3]]),
    (([1, 2, 3], 3, 1), [[1], [2], [3]]),
    (([1, 2, 3, 4], 3, 2), ValueError),
    (([], 0, 0), ValueError),
]
FLATTEN_CASES = [
    (([[1, 2, 3], [4, 5, 6]],), [1, 2, 3, 4, 5, 6]),
    (([[7]],), [7]),
    (([[1], [2]],), [1, 2]),
    (([[1, 2], [3]],), ValueError),
]
DIAGONAL_CASES = [
    (([1, 2, 3],), [[1, 0, 0], [0, 2, 0], [0, 0, 3]]),
    (([5],), [[5]]),
    (([0.5, -1],), [[0.5, 0], [0, -1]]),
    (([],), ValueError),
]
check("reshape", reshape, RESHAPE_CASES)
check("flatten", flatten, FLATTEN_CASES)
check("diagonal_matrix", diagonal_matrix, DIAGONAL_CASES)

### Exercise 4: examine rounding errors

* `relative_error(approximate, exact)` returns $\dfrac{|\text{approximate} - \text{exact}|}{|\text{exact}|}$. Raise `ValueError` when `exact` is zero, because the relative error is then undefined. These tests use no absolute tolerance, so even an error of about $10^{-16}$ must come out right.
* `approx_equal(a, b, rel_tol=1e-9, abs_tol=0.0)` re-implements the rule behind `math.isclose`: return `True` when $|a - b| \le \max(r \cdot \max(|a|, |b|),\ t)$, where $r$ is `rel_tol` and $t$ is `abs_tol`. Write the comparison yourself instead of calling `math.isclose`.
* `float_predictions()` returns six booleans: your predictions, **made before running anything**, for the six expressions listed in its docstring. Afterwards, run the expressions in a new cell, and use the theory to explain any that surprised you.

In [ ]:
def relative_error(approximate, exact):
    """|approximate - exact| / |exact|; raises ValueError when exact is zero."""
    # TODO: guard against exact == 0, then return the relative error.
    raise NotImplementedError("Exercise 4a: implement relative_error")


def approx_equal(a, b, rel_tol=1e-9, abs_tol=0.0):
    """True when |a - b| <= max(rel_tol * max(|a|, |b|), abs_tol)."""
    # TODO: translate the inequality directly; do not call math.isclose.
    raise NotImplementedError("Exercise 4b: implement approx_equal")


def float_predictions():
    """Predict, without running them, whether each expression is True or False.

    1. 0.1 + 0.2 == 0.3
    2. 0.5 + 0.25 == 0.75
    3. (1e16 + 1.0) - 1e16 == 1.0
    4. round(2.675, 2) == 2.68
    5. 2 ** 53 + 1 == 2 ** 53
    6. 2.0 ** 53 + 1 == 2.0 ** 53

    Return the six predictions as a tuple of booleans, in this order.
    """
    # TODO: replace the line below with your answer, for example: return (True, False, ...)
    raise NotImplementedError("Exercise 4c: record your predictions")

In [ ]:
RELATIVE_ERROR_CASES = [
    ((1.1, 1.0), 0.1),
    ((99.0, 100.0), 0.01),
    ((-1.5, -1.0), 0.5),
    ((2.0, 2.0), 0.0),
    ((0.1 + 0.2, 0.3), 1.8503717077085943e-16),  # one rounding step, relative to 0.3
    ((5.0, 0.0), ValueError),
]
APPROX_EQUAL_CASES = [
    ((0.1 + 0.2, 0.3), True),
    ((1.0, 1.1), False),
    ((1e-20, 0.0), False),
    ((1e-20, 0.0, 1e-9, 1e-12), True),
    ((1e10, 1e10 + 1), True),
    ((100.0, 101.0, 0.02), True),
    ((100.0, 103.0, 0.02), False),
    ((-1.0, 1.0), False),
]
FLOAT_PREDICTION_CASES = [
    ((), (False, True, False, False, False, True)),
]
check("relative_error", relative_error, RELATIVE_ERROR_CASES, abs_tol=0.0)
check("approx_equal", approx_equal, APPROX_EQUAL_CASES)
check("float_predictions", float_predictions, FLOAT_PREDICTION_CASES)

When `float_predictions` passes, compare your reasoning with the comments in its reference solution at the end of the notebook.

## Common errors and debugging guidance

| Symptom | Likely cause | What to do |
|---|---|---|
| `0.1 + 0.2 == 0.3` is `False`, or a float test fails by about $10^{-16}$ | exact comparison of rounded values | compare with `math.isclose`, and add `abs_tol` when one side can be `0` |
| `-3 ** 2` gives `-9`, or `1 / 2 * x` is not $\frac{1}{2x}$ | operator precedence | add parentheses: `(-3) ** 2`, `1 / (2 * x)` |
| a sum misses its last term | `range(1, n)` stops at `n - 1` | use `range(1, n + 1)`, and remember that Python indices start at 0 |
| `RecursionError` | no base case, or a base case that is never reached | check the base case first, and prefer a loop for deep computations |
| `TypeError` that mentions `'NoneType'` | a function forgot to `return` | make sure every path through the function returns a value |
| changing one row of a matrix changes the others | rows built with `[[0] * n] * m` share one list | build rows with `[[0] * n for _ in range(m)]` |
| `OverflowError`, or an unexpected `inf` | a value beyond about $1.8 \times 10^{308}$ | rearrange the formula, as `sigmoid` does, or work with logarithms |
| a long sum drifts | accumulated rounding errors | use `math.fsum` |
| the difference of two large, close numbers is wildly wrong | catastrophic cancellation | subtract before squaring (two-pass variance); use `math.log1p` and `math.expm1` near zero |
| `Vector2D(1, 2) == Vector2D(1, 2)` is `False` | without `__eq__`, `==` compares identity | compare the components, or define `__eq__` |

**Debugging habits for mathematical code**

1. **Restart and run everything.** A notebook remembers old definitions, so a cell can work only because of a cell you have since deleted. Use *Kernel → Restart Kernel and Run All Cells* before you trust a result.
2. **Test a case you can do by hand.** Before trusting `variance` on real data, try `[4, 7, 13, 16]`, whose variance you know is $22.5$.
3. **Print `repr(x)` and `type(x)`.** `repr` shows enough digits to identify a float exactly, and `type` reveals an `int` where you expected a `float`, or `None` where you expected a number.
4. **Look at the exact value.** `Fraction(x)` and `Decimal(x)` show exactly which number a float holds.
5. **Read the `check` message.** It names the call that failed, what your function returned and what was expected. Work that one case out by hand.
6. **Give your asserts messages.** `assert condition, "what should be true"` records your intent and makes a failure explain itself.

## Summary

* Python integers are exact at any size. Floats are binary approximations with a 53-bit significand, a machine epsilon of $2^{-52} \approx 2.2 \times 10^{-16}$ and a finite range.
* `/` always returns a float, `//` rounds down towards $-\infty$, and `**` binds more tightly than unary minus.
* Functions translate formulas, loops with an accumulator translate $\sum$ and $\prod$, and recursion needs a base case. The empty sum is $0$ and the empty product is $1$.
* A matrix is a list of equally long rows. Validate shapes, index from 0, and never build rows with `[[0] * n] * m`.
* Special methods such as `__add__`, `__mul__`, `__rmul__` and `__repr__` let your own classes use mathematical notation.
* Expect rounding: compare floats with `math.isclose` (with `abs_tol` near zero), add many floats with `math.fsum`, and rearrange formulas that subtract nearly equal numbers.

**Next:** Lesson 2 turns these lists into vectors and matrices and implements the operations at the heart of every linear model.

## Solutions

Try every exercise before you read this section. Each reference solution has a `_solution` suffix, so it never replaces your own function. Each one runs through the same `check` cases with `strict=True`, which turns `not implemented yet` into a failure, so passing here shows that the tests themselves are correct.

In [ ]:
def polynomial_solution(coefficients, x):
    result = 0
    for c in reversed(coefficients):  # Horner: c0 + x(c1 + x(c2 + ...))
        result = result * x + c
    return result


def n_choose_k_solution(n, k):
    if n < 0 or k < 0:
        raise ValueError(f"n and k must be non-negative, got n={n}, k={k}")
    if k > n:
        return 0
    k = min(k, n - k)  # C(n, k) = C(n, n - k), and the smaller k needs fewer steps
    result = 1
    for i in range(1, k + 1):
        result = result * (n - k + i) // i  # an exact integer after every step
    return result


check("polynomial_solution", polynomial_solution, POLYNOMIAL_CASES, strict=True)
check("n_choose_k_solution", n_choose_k_solution, N_CHOOSE_K_CASES, strict=True)

In [ ]:
def mean_solution(values):
    if len(values) == 0:
        raise ValueError("the mean of an empty list is undefined")
    return math.fsum(values) / len(values)


def variance_solution(values):
    centre = mean_solution(values)  # raises ValueError for an empty list
    return math.fsum((x - centre) ** 2 for x in values) / len(values)


def mean_squared_error_solution(predictions, targets):
    if len(predictions) != len(targets):
        raise ValueError(f"got {len(predictions)} predictions but {len(targets)} targets")
    if len(predictions) == 0:
        raise ValueError("the mean squared error of empty lists is undefined")
    return math.fsum((p - t) ** 2 for p, t in zip(predictions, targets)) / len(predictions)


check("mean_solution", mean_solution, MEAN_CASES, strict=True)
check("variance_solution", variance_solution, VARIANCE_CASES, strict=True)
check("mean_squared_error_solution", mean_squared_error_solution, MSE_CASES, strict=True)

In [ ]:
def reshape_solution(values, rows, columns):
    if rows < 1 or columns < 1:
        raise ValueError(f"rows and columns must be at least 1, got {rows} and {columns}")
    if len(values) != rows * columns:
        raise ValueError(f"cannot arrange {len(values)} values as {rows} x {columns}")
    return [list(values[r * columns:(r + 1) * columns]) for r in range(rows)]


def flatten_solution(matrix):
    shape(matrix)  # raises ValueError for empty or ragged input
    return [entry for row in matrix for entry in row]


def diagonal_matrix_solution(diagonal):
    n = len(diagonal)
    if n == 0:
        raise ValueError("a diagonal matrix needs at least one entry")
    result = [[0] * n for _ in range(n)]  # a separate list for every row
    for i, value in enumerate(diagonal):
        result[i][i] = value
    return result


check("reshape_solution", reshape_solution, RESHAPE_CASES, strict=True)
check("flatten_solution", flatten_solution, FLATTEN_CASES, strict=True)
check("diagonal_matrix_solution", diagonal_matrix_solution, DIAGONAL_CASES, strict=True)

In [ ]:
def relative_error_solution(approximate, exact):
    if exact == 0:
        raise ValueError("the relative error is undefined when the exact value is zero")
    return abs(approximate - exact) / abs(exact)


def approx_equal_solution(a, b, rel_tol=1e-9, abs_tol=0.0):
    return abs(a - b) <= max(rel_tol * max(abs(a), abs(b)), abs_tol)


def float_predictions_solution():
    return (
        False,  # 1. 0.1, 0.2 and 0.3 are all rounded, and the sum lands one float above 0.3
        True,   # 2. 0.5, 0.25 and 0.75 are exact binary fractions, so nothing is rounded
        False,  # 3. 1e16 + 1.0 rounds back to 1e16, because neighbouring floats are 2 apart
        False,  # 4. 2.675 is stored as 2.67499999..., so it rounds down to 2.67
        False,  # 5. integers are exact, so 2 ** 53 + 1 really is one more than 2 ** 53
        True,   # 6. as floats, 2 ** 53 + 1 rounds back to 2 ** 53
    )


check("relative_error_solution", relative_error_solution, RELATIVE_ERROR_CASES, abs_tol=0.0, strict=True)
check("approx_equal_solution", approx_equal_solution, APPROX_EQUAL_CASES, strict=True)
check("float_predictions_solution", float_predictions_solution, FLOAT_PREDICTION_CASES, strict=True)

**Expected behaviour.** Every solution prints a pass line, such as `variance_solution: all 5 tests passed`. If you change a solution so that it is wrong, its `check` call raises `AssertionError`, which shows that the tests can tell right answers from wrong ones.